[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 中等：实现 BatchNorm

实现带有**训练**和**推理**两种模式的**批归一化（Batch Normalization）**。

训练模式下，使用**批统计量**并更新滑动估计：

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

其中 $\mu_B$ 和 $\sigma_B^2$ 是在**批次维度**（dim=0）上计算的均值和方差。

推理模式下，使用提供的**滑动均值/方差**替代当前批次的统计量。

### 函数签名
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — 对批次中所有样本的每个特征进行归一化
    # running_mean, running_var: 训练时原地更新；推理时直接使用
```

### 规则
- **不得**使用 `F.batch_norm`、`nn.BatchNorm1d` 等
- 使用 `unbiased=False` 在 `dim=0` 上计算批次均值和方差
- 按 PyTorch 方式更新滑动统计量：`running = (1 - momentum) * running + momentum * batch_stat`
- 当 `training=False` 时使用 `running_mean` / `running_var` 进行推理
- 必须支持对 `x`、`gamma`、`beta` 的自动求导（滑动统计量视为缓冲区，不需要梯度）

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    pass  # Replace this

In [ ]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm")